# Personal expense and incoming-money analysis
Run after reviewing your statement and confirming complete dates in the UI. No private data is bundled. Do not commit executed outputs. Questionnaire, convenience sampling and Google Form collection are not satisfied by transaction records.

In [ ]:
import core
import pandas as pd
import matplotlib.pyplot as plt
tx = pd.DataFrame(core.rows())
print("Rows, columns:", tx.shape)
display(tx.dtypes)
display(tx.isna().sum())
print("Duplicate IDs:", tx.id.duplicated().sum())

## Daily data and EDA
Only contiguous confirmed days are included. Missing entries must be corrected before confirmation. Valid outliers are retained. Review patterns rather than claiming causes.

In [ ]:
daily = core.daily_history()
display(daily.describe())
daily.hist(figsize=(10,4)); plt.show()
daily.plot.box(); plt.show()
display(daily.corr())
daily.plot.scatter(x="credit", y="expense"); plt.show()
display(tx.groupby(["category", "direction"]).amount.agg(["count", "sum"]))

## Features and leakage control
Lagged and shifted rolling values use only past dates. Weekday uses circular encoding. Ridge standardizes within training folds; no categorical variable requires one-hot encoding here.

In [ ]:
features = core.features(daily).dropna()
display(features.head())
print("Modelling dates:", len(features))
corr = features.join(daily.expense.rename("target")).corr()["target"].drop("target")
display(pd.DataFrame({"correlation":corr,"strength":corr.abs().map(lambda v: "high" if v>=.7 else "moderate" if v>=.3 else "low")}))

## Train and compare
This updates the models used by the website. Do not run while the website is training. Model selection is by time-based validation, not test R². A separate classifier is unnecessary for the exact budget comparison.

In [ ]:
result = core.train()
display(pd.DataFrame(result["comparison"]))
print(result["note"])

## Seven-day prediction and category allocation

In [ ]:
end = pd.Timestamp(core.history_end())
prediction = core.forecast(str((end + pd.Timedelta(days=7)).date()))
display(pd.DataFrame(prediction["days"]).drop(columns="categories"))
display(pd.DataFrame(prediction["category_breakdown"]))
print(prediction["cashflow"])
print(prediction["note"])